# Explainer figure 4: SAI precipitation response by downscaling method

Maps of the G6-1.5K minus SSP2-4.5 change in annual precipitation for the raw GCM, BCSD, and QDMSD, plus area-weighted means over the Mekong River Basin.

Outputs go to `s3://carbonplan-srm/output/explainer/figures/figure4/`:

- `annual_{method}.zarr`: global annual-mean precipitation per scenario and member, 2035–2084. Building these reads about 0.5 TB of daily data and runs on Coiled.
- `{method}.zarr`: global `delta_mm` and `delta_pct` maps rendered by the web figure.
- `figure4.json` (basin statistics and map extents) and `mekong.json` (basin outline), which the article's `MethodComparison` component imports as local copies.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import geopandas as gpd
import icechunk
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from shapely import contains_xy

from saidownscale.config import ClusterConfig, setup_cluster

In [ ]:
FIGURE_ROOT = "s3://carbonplan-srm/output/explainer/figures/figure4"

GCM = "CESM2-WACCM6"
SOURCE_COOP_BUCKET = "us-west-2.opendata.source.coop"
SOURCE_COOP_ROOT = "carbonplan/srm-downscaling"
INPUT_STORE = "input/processed/CESM2-WACCM6.icechunk"
PROD_STORE = "output/production/CESM2-WACCM6-ERA5-global.icechunk"
PROD_BRANCH = "v1.0.0"

SCENARIOS = ["ssp245", "g6_1p5k"]
METHODS = ["raw", "bcsd", "qdmsd"]
MEMBERS = ["001", "002", "003"]  # members with both scenarios in v1.0.0
ANNUAL_YEARS = slice("2035", "2084")  # full G6-1.5K extent
PERIOD = slice(2055, 2084)  # averaging window

KG_M2_S_TO_MM_YEAR = 86400 * 365
PCT_MASK_THRESHOLD = 1e-7  # kg m-2 s-1; percent change is masked where SSP245 is near zero

REGION_NAME = "Mekong River Basin"
REGION_FILE = "data/mekong_basin.geojson"
VIEW_PAD_DEG = 3.0

OVERWRITE = False  # recompute annual stores that already exist

## Choices

**GCM.** CESM2-WACCM6, matching figures 1 and 3. In v1.0.0 it has both scenarios, both downscaling methods, and the same three members for each, so the panels (raw GCM, BCSD, QDMSD) differ only in how the data were downscaled.

**Members.** 001, 002, and 003, the only members with a G6-1.5K run. Each is differenced against its own SSP2-4.5 run, then the three differences are averaged.

**Period.** 2055–2084, the last 30 years of G6-1.5K. The annual stores cover 2035–2084, so the window can change without recomputing.

## 1. Annual means per member (Coiled)

Daily precipitation is reduced to annual means on the cluster and written to S3.

In [4]:
client = setup_cluster(ClusterConfig(spot_policy="on-demand"))
client

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                    ╷                                                         │
│   Package          │ Note                                                    │
│ ╶──────────────────┼───────────────────────────────────────────────────────╴ │
│   coiled_local_src │ Source wheel built from                                 │
│                    │ ~/dev/carbonplan/srm-downscaling/src                    │
│   srm              │ Wheel built from ~/dev/carbonplan/srm-downscaling       │
│                    ╵                                                         │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

2026-09-14 21:34:38,649 - distributed.deploy.adaptive - INFO - Adaptive scaling started: minimum=1 maximum=50


<Client: 'tls://10.1.37.127:8786' processes=1 threads=8, memory=60.66 GiB>

In [ ]:
def source_coop_session(store, branch="main"):
    storage = icechunk.s3_storage(
        bucket=SOURCE_COOP_BUCKET,
        prefix=f"{SOURCE_COOP_ROOT}/{store}",
        anonymous=True,
        region="us-west-2",
    )
    return icechunk.Repository.open(storage).readonly_session(branch=branch)


input_session = source_coop_session(INPUT_STORE)
prod_session = source_coop_session(PROD_STORE, branch=PROD_BRANCH)


def daily_pr(method, scenario, member):
    if method == "raw":
        ds = xr.open_zarr(
            input_session.store, group=scenario, consolidated=False, zarr_format=3, chunks={}
        )
        return ds["pr"].sel(ensemble_member=member, drop=True)
    return xr.open_zarr(
        prod_session.store,
        group=f"{method}/{scenario}/pr/{member}",
        consolidated=False,
        zarr_format=3,
        chunks={"time": 365, "lat": 180, "lon": 360},
    )["pr"]


def annual_means(method):
    per_scenario = []
    for scenario in SCENARIOS:
        per_member = [
            daily_pr(method, scenario, m).sel(time=ANNUAL_YEARS).groupby("time.year").mean("time")
            for m in MEMBERS
        ]
        per_scenario.append(xr.concat(per_member, dim=pd.Index(MEMBERS, name="ensemble_member")))
    da = xr.concat(per_scenario, dim=pd.Index(list(SCENARIOS), name="scenario")).rename("pr")
    da.attrs = {
        "units": "kg m-2 s-1",
        "long_name": "Annual mean precipitation",
        "source": "raw GCM" if method == "raw" else f"{method} {PROD_BRANCH}",
        "gcm": GCM,
    }
    return da


def spatial_chunks(da):
    if da.sizes["lat"] < 400:
        return {"lat": -1, "lon": -1}
    return {"lat": 181, "lon": 180}


def store_complete(store):
    try:
        arr = zarr.open_array(store, path="pr", mode="r")
    except (FileNotFoundError, zarr.errors.ArrayNotFoundError):
        return False
    return arr.nchunks_initialized == arr.nchunks


for method in METHODS:
    store = f"{FIGURE_ROOT}/annual_{method}.zarr"
    if not OVERWRITE and store_complete(store):
        print("exists", store)
        continue
    da = annual_means(method)
    da = da.chunk({"scenario": 1, "ensemble_member": 1, "year": -1, **spatial_chunks(da)})
    da.drop_encoding().to_zarr(store, mode="w", zarr_format=3)
    print("wrote", method, dict(da.sizes))

## 2. Difference maps

Annual means are averaged over members and the period, then differenced between scenarios. The maps are global and spatially chunked, so the web figure only fetches the chunks in view.

In [6]:
def open_annual(method):
    return xr.open_zarr(f"{FIGURE_ROOT}/annual_{method}.zarr", zarr_format=3)["pr"]


def edge_bounds(da):
    lat, lon = da.lat.values, da.lon.values
    dlat, dlon = abs(lat[1] - lat[0]) / 2, abs(lon[1] - lon[0]) / 2
    return [
        float(lon.min() - dlon),
        float(max(lat.min() - dlat, -90)),
        float(lon.max() + dlon),
        float(min(lat.max() + dlat, 90)),
    ]


bounds = {}
for method in METHODS:
    pr = open_annual(method).sel(year=PERIOD).mean(["year", "ensemble_member"])
    g6, ssp = pr.sel(scenario="g6_1p5k"), pr.sel(scenario="ssp245")
    delta = g6 - ssp
    maps = xr.Dataset(
        {
            "delta_mm": (delta * KG_M2_S_TO_MM_YEAR).assign_attrs(units="mm/yr"),
            "delta_pct": (100 * delta / ssp)
            .where(ssp >= PCT_MASK_THRESHOLD)
            .assign_attrs(units="%"),
        }
    ).astype("float32")
    maps.attrs = {
        "description": f"G6-1.5K minus SSP2-4.5 annual precipitation, {PERIOD.start}-{PERIOD.stop} mean",
        "ensemble_members": MEMBERS,
        "source": str(pr.attrs.get("source", "")),
    }
    bounds[method] = edge_bounds(maps)
    maps = maps.sortby("lat").chunk(spatial_chunks(maps))
    maps.drop_encoding().to_zarr(f"{FIGURE_ROOT}/{method}.zarr", mode="w", zarr_format=3)
    print("wrote", method, bounds[method])

/Users/shaneloeffler/dev/carbonplan/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=7, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


/Users/shaneloeffler/dev/carbonplan/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote raw [-180.625, -90.0, 179.375, 90.0]


/Users/shaneloeffler/dev/carbonplan/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=7, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


/Users/shaneloeffler/dev/carbonplan/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote bcsd [-180.125, -90.0, 179.875, 90.0]


/Users/shaneloeffler/dev/carbonplan/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=7, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


/Users/shaneloeffler/dev/carbonplan/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote qdmsd [-180.125, -90.0, 179.875, 90.0]


In [7]:
client.close()

## 3. Basin statistics

Area-weighted basin means for each member. A cell counts when its center falls inside the basin. The basin outline is MRBID 2421 from GRDC's Major River Basins of the World (2nd ed., 2020), simplified to 0.01°.

In [8]:
region = gpd.read_file(REGION_FILE)
geom = region.geometry.iloc[0]
minx, miny, maxx, maxy = geom.bounds
view = [
    minx - VIEW_PAD_DEG,
    max(miny - VIEW_PAD_DEG, -90),
    maxx + VIEW_PAD_DEG,
    min(maxy + VIEW_PAD_DEG, 90),
]


def basin_mean(da):
    lon2d, lat2d = np.meshgrid(da.lon.values, da.lat.values)
    inside = contains_xy(geom, lon2d, lat2d)
    weights = xr.DataArray(np.where(inside, np.cos(np.deg2rad(lat2d)), 0.0), dims=("lat", "lon"))
    return da.weighted(weights).mean(["lat", "lon"]), int(inside.sum())


basin = {}
for method in METHODS:
    annual = (
        open_annual(method)
        .sel(year=PERIOD, lat=slice(miny - 1, maxy + 1), lon=slice(minx - 1, maxx + 1))
        .load()
    )
    series, n_cells = basin_mean(annual)
    period_mean = series.mean("year")
    g6, ssp = period_mean.sel(scenario="g6_1p5k"), period_mean.sel(scenario="ssp245")
    mm = (g6 - ssp) * KG_M2_S_TO_MM_YEAR
    pct = 100 * (g6 - ssp) / ssp
    basin[method] = {
        "n_cells": n_cells,
        "mean": {"mm": float(mm.mean()), "pct": float(pct.mean())},
        "members": [
            {
                "member": m,
                "mm": float(mm.sel(ensemble_member=m)),
                "pct": float(pct.sel(ensemble_member=m)),
            }
            for m in MEMBERS
        ],
    }

pd.DataFrame({m: basin[m]["mean"] | {"n_cells": basin[m]["n_cells"]} for m in METHODS}).T

,mm,pct,n_cells
raw,-85.580026,-4.537555,54.0
bcsd,-54.973385,-3.192726,1071.0
qdmsd,-70.345619,-4.100200,1071.0


## 4. Export JSON for the figure

In [9]:
import s3fs

fs = s3fs.S3FileSystem()

figure = {
    "region": REGION_NAME,
    "gcm": GCM,
    "period": [PERIOD.start, PERIOD.stop],
    "members": MEMBERS,
    "view": view,
    "bounds": bounds,
    "basin": basin,
}
with fs.open(f"{FIGURE_ROOT}/figure4.json", "w") as f:
    json.dump(figure, f, indent=2)

with fs.open(f"{FIGURE_ROOT}/mekong.json", "w") as f:
    f.write(region[["name", "geometry"]].to_json())

print(json.dumps(figure, indent=2))

{
  "region": "Mekong River Basin",
  "gcm": "CESM2-WACCM6",
  "period": [
    2055,
    2084
  ],
  "members": [
    "001",
    "002",
    "003"
  ],
  "view": [
    90.85806867819491,
    6.528599378928334,
    111.77048000767098,
    36.82083333309902
  ],
  "bounds": {
    "raw": [
      -180.625,
      -90.0,
      179.375,
      90.0
    ],
    "bcsd": [
      -180.125,
      -90.0,
      179.875,
      90.0
    ],
    "qdmsd": [
      -180.125,
      -90.0,
      179.875,
      90.0
    ]
  },
  "basin": {
    "raw": {
      "n_cells": 54,
      "mean": {
        "mm": -85.58002593642671,
        "pct": -4.537554627047623
      },
      "members": [
        {
          "member": "001",
          "mm": -82.03953325377549,
          "pct": -4.324060257721956
        },
        {
          "member": "002",
          "mm": -56.90759039486759,
          "pct": -3.0566729545819653
        },
        {
          "member": "003",
          "mm": -117.7929541606371,
          "pct": -6.2

The web figure fetches the map stores anonymously, and the bucket policy only exposes `input/` and `output/production/`, so they get a public-read ACL, as figure 1's stores do.

In [10]:
import boto3

s3 = boto3.client("s3")
bucket, prefix = FIGURE_ROOT.replace("s3://", "").split("/", 1)
public = [f"{prefix}/{m}.zarr/" for m in METHODS]

for key_prefix in public:
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=key_prefix):
        for obj in page.get("Contents", []):
            s3.put_object_acl(Bucket=bucket, Key=obj["Key"], ACL="public-read")
print("public-read set on", public)

public-read set on ['output/explainer/figures/figure4/raw.zarr/', 'output/explainer/figures/figure4/bcsd.zarr/', 'output/explainer/figures/figure4/qdmsd.zarr/']
